In [1]:
import pandas as pd

from label_mapper import (map_cbis, map_rsna, map_vindr, to_int)

## Load Datasets

Load metadata from all datasets:

- **CBIS-DDSM** → pathology-based labels
- **RSNA** → cancer (0/1)
- **VinDr-Mammo** → BI-RADS categories

In [2]:
cbis = pd.read_csv("cbis_ddsm_metadata.csv")
rsna = pd.read_csv("rsna_metadata.csv")
vindr = pd.read_csv("vindr_metadata.csv")

## CBIS-DDSM Label Mapping

CBIS-DDSM uses pathology labels derived from biopsy results.

### Mapping:
- MALIGNANT → WORTH_SECOND_LOOK
- BENIGN → WORTH_SECOND_LOOK *(biopsied → suspicious)*
- BENIGN_WITHOUT_CALLBACK → NOT_WORTH_SECOND_LOOK

In [3]:
cbis["label"] = cbis["pathology"].apply(map_cbis)
cbis["label"].value_counts()

label
Label.WORTH_SECOND_LOOK        1214
Label.NOT_WORTH_SECOND_LOOK     104
Name: count, dtype: int64

## RSNA Label Mapping

RSNA provides screening-level cancer labels.

### Mapping:
- 1 → WORTH_SECOND_LOOK
- 0 → NOT_WORTH_SECOND_LOOK


In [4]:
rsna["label"] = rsna["cancer"].apply(map_rsna)
rsna["label"].value_counts()

label
Label.NOT_WORTH_SECOND_LOOK    53548
Label.WORTH_SECOND_LOOK         1158
Name: count, dtype: int64

## VinDr-Mammo Label Mapping

VinDr uses BI-RADS categories assigned by radiologists.

### Mapping:
- BI-RADS 1–2 → NOT_WORTH_SECOND_LOOK
- BI-RADS 3–6 → WORTH_SECOND_LOOK

BI-RADS 3 is included as positive to prioritize sensitivity.

- `finding_birads` is **lesion-level**
- `breast_birads` is **breast-level (recommended)**

Normal cases may appear as missing values in lesion-level data

In [5]:
vindr["label"] = vindr["breast_birads"].apply(map_vindr)
vindr["label"].value_counts()

label
Label.NOT_WORTH_SECOND_LOOK    18082
Label.WORTH_SECOND_LOOK         2404
Name: count, dtype: int64

## Convert Labels to Training Targets

Convert unified `Label` values into numeric targets for model training.

### Mapping:
- `WORTH_SECOND_LOOK` → 1
- `NOT_WORTH_SECOND_LOOK` → 0

This step ensures compatibility with machine learning frameworks
such as TensorFlow and PyTorch.

Each dataset now contains:
- `label` → human-readable Enum
- `target` → numeric value used for training

In [6]:
cbis["target"] = cbis["label"].apply(to_int)
rsna["target"] = rsna["label"].apply(to_int)
vindr["target"] = vindr["label"].apply(to_int)

In [7]:
cbis["target"].value_counts()
rsna["target"].value_counts()
vindr["target"].value_counts()

target
0    18082
1     2404
Name: count, dtype: int64